In [1]:
# FP-Growth Modeling for Association Rules
# Bước 3b: Khai phá luật kết hợp bằng thuật toán FP-Growth

#**Mục tiêu:**
# - Tìm tập mục phổ biến bằng thuật toán FP-Growth
# - Sinh luật kết hợp với support, confidence, lift
# - So sánh hiệu suất với Apriori (trong notebook riêng)

# **Pipeline:**
# 1. Tải basket_bool từ bước 2
# 2. Chạy FP-Growth để tìm frequent itemsets
# 3. Sinh association rules
# 4. Lọc luật theo ngưỡng
# 5. Trực quan hoá kết quả
# 6. Lưu luật để so sánh

In [2]:
# PARAMETERS (cho papermill)

# Đường dẫn dữ liệu
BASKET_BOOL_PATH = "data/processed/basket_bool.parquet"
RULES_OUTPUT_PATH = "data/processed/rules_fpgrowth_filtered.csv"

# Tham số FP-Growth
MIN_SUPPORT = 0.01
MAX_LEN = 3

# Tham số sinh luật
METRIC = "lift"
MIN_THRESHOLD = 1.0

# Tham số lọc luật
FILTER_MIN_SUPPORT = 0.01
FILTER_MIN_CONF = 0.3
FILTER_MIN_LIFT = 1.2
FILTER_MAX_ANTECEDENTS = 2
FILTER_MAX_CONSEQUENTS = 1

# Visualization
TOP_N_RULES = 20
PLOT_TOP_LIFT = True
PLOT_TOP_CONF = True
PLOT_SCATTER = True
PLOT_NETWORK = True
PLOT_PLOTLY_SCATTER = True

print("✅ Parameters loaded for FP-Growth")

✅ Parameters loaded for FP-Growth


In [3]:
# Parameters
BASKET_BOOL_PATH = "data/processed/basket_bool.parquet"
RULES_OUTPUT_PATH = "data/processed/rules_fpgrowth_filtered.csv"
MIN_SUPPORT = 0.05
MAX_LEN = 3
METRIC = "lift"
MIN_THRESHOLD = 1.0
FILTER_MIN_SUPPORT = 0.01
FILTER_MIN_CONF = 0.3
FILTER_MIN_LIFT = 1.2
FILTER_MAX_ANTECEDENTS = 2
FILTER_MAX_CONSEQUENTS = 1
TOP_N_RULES = 20
PLOT_TOP_LIFT = False
PLOT_TOP_CONF = False
PLOT_SCATTER = False
PLOT_NETWORK = False
PLOT_PLOTLY_SCATTER = False


In [4]:
# Setup environment
%load_ext autoreload
%autoreload 2

import os
import sys
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Setup visualization
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

# Fix path issues
current_dir = os.getcwd()
print(f"📂 Current directory: {current_dir}")

# If running from notebooks folder
if os.path.basename(current_dir) == "notebooks":
    project_root = os.path.abspath("..")
    print(f"📁 Project root (auto-detected): {project_root}")
else:
    project_root = current_dir
    print(f"📁 Project root: {project_root}")

# Add src to path
src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)
    print(f"📦 Added to path: {src_path}")

# Import custom library
try:
    from apriori_library import FPGrowthMiner, DataVisualizer
    print("✅ Successfully imported FPGrowthMiner and DataVisualizer")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Trying to import all...")
    from apriori_library import *

📂 Current directory: E:\Data Mining\Nhom7-CNTT17_10-DataMining
📁 Project root: E:\Data Mining\Nhom7-CNTT17_10-DataMining
📦 Added to path: E:\Data Mining\Nhom7-CNTT17_10-DataMining\src


✅ Successfully imported FPGrowthMiner and DataVisualizer


In [5]:
import os
import pandas as pd
import numpy as np

print("🔍 Checking file paths...")

basket_bool = None  # 🔒 đảm bảo biến luôn tồn tại

if not os.path.exists(BASKET_BOOL_PATH):
    print(f"⚠️  Relative path not found: {BASKET_BOOL_PATH}")

    abs_basket_path = os.path.join(
        project_root, "data", "processed", "basket_bool.parquet"
    )
    print(f"   Trying absolute path: {abs_basket_path}")

    if os.path.exists(abs_basket_path):
        BASKET_BOOL_PATH = abs_basket_path
        print(f"✅ Found at: {BASKET_BOOL_PATH}")
        basket_bool = pd.read_parquet(BASKET_BOOL_PATH)   # ✅ ĐỌC FILE
    else:
        print("❌ File not found. Please run basket_preparation.ipynb first.")
        print("⚠️  Creating dummy data for testing...")

        np.random.seed(42)
        basket_bool = pd.DataFrame(
            np.random.choice([0, 1], size=(1000, 100), p=[0.97, 0.03]),
            columns=[f"Product_{i:03d}" for i in range(1, 101)]
        )
        basket_bool.index.name = "InvoiceNo"
else:
    print(f"✅ Found basket_bool at: {BASKET_BOOL_PATH}")
    basket_bool = pd.read_parquet(BASKET_BOOL_PATH)

# ✅ an toàn tuyệt đối
print(f"\n📊 Basket bool shape: {basket_bool.shape}")
print(f"📈 Density: {basket_bool.values.mean():.4%}")
print(f"💾 Memory usage: {basket_bool.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\n👀 First 5 invoices, first 8 products:")
display(basket_bool.iloc[:5, :8])


🔍 Checking file paths...
✅ Found basket_bool at: data/processed/basket_bool.parquet



📊 Basket bool shape: (18021, 4007)
📈 Density: 0.6573%


💾 Memory usage: 68.87 MB

👀 First 5 invoices, first 8 products:


,4 PURPLE FLOCK DINNER CANDLES,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,NINE DRAWER OFFICE TIDY,OVAL WALL MIRROR DIAMANTE,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON
0,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False


In [6]:
# Khởi tạo FP-Growth miner
print("🚀 Initializing FP-Growth Miner...")
miner = FPGrowthMiner(basket_bool=basket_bool)
print("✅ FP-Growth Miner ready!")

🚀 Initializing FP-Growth Miner...
✅ FP-Growth Miner ready!


In [7]:
print("\n" + "="*60)
print("MINING FREQUENT ITEMSETS WITH FP-GROWTH")
print("="*60)

# Ghi lại thời gian bắt đầu
start_time = time.time()

# Chạy FP-Growth
frequent_itemsets_fp = miner.mine_frequent_itemsets(
    min_support=MIN_SUPPORT,
    max_len=MAX_LEN,
    use_colnames=True
)

# Tính thời gian
fp_time = time.time() - start_time
print(f"\n⏱️  FP-Growth completed in {fp_time:.2f} seconds")
print(f"📦 Found {len(frequent_itemsets_fp):,} frequent itemsets")

# Hiển thị thống kê
if 'length' in frequent_itemsets_fp.columns:
    length_dist = frequent_itemsets_fp['length'].value_counts().sort_index()
    print("\n📐 Itemset length distribution:")
    for length, count in length_dist.items():
        print(f"   {length}-itemsets: {count:,}")

# Top itemsets
print("\n🏆 Top 10 frequent itemsets:")
top_itemsets = frequent_itemsets_fp.sort_values('support', ascending=False).head(10)
display(top_itemsets[['itemsets', 'support']])


MINING FREQUENT ITEMSETS WITH FP-GROWTH
FP-GROWTH ALGORITHM
Parameters: min_support=0.05, max_len=3


✓ Completed in 3.56 seconds
✓ Found 34 frequent itemsets

⏱️  FP-Growth completed in 3.56 seconds
📦 Found 34 frequent itemsets

📐 Itemset length distribution:
   1-itemsets: 34

🏆 Top 10 frequent itemsets:


,itemsets,support
0,(WHITE HANGING HEART T-LIGHT HOLDER),0.119971
16,(JUMBO BAG RED RETROSPOT),0.107375
23,(REGENCY CAKESTAND 3 TIER),0.093502
28,(PARTY BUNTING),0.088397
6,(LUNCH BAG RED RETROSPOT),0.077243
1,(ASSORTED COLOUR BIRD ORNAMENT),0.076078
29,(SET OF 3 CAKE TINS PANTRY DESIGN ),0.068864
13,(NATURAL SLATE HEART CHALKBOARD ),0.067643
20,(LUNCH BAG BLACK SKULL.),0.067477
14,(HEART OF WICKER SMALL),0.064591


In [8]:
print("\n" + "="*60)
print("GENERATING ASSOCIATION RULES")
print("="*60)

# Sinh luật
rules_fp = miner.generate_rules(
    metric=METRIC,
    min_threshold=MIN_THRESHOLD
)

# Thêm cột dễ đọc
rules_fp = miner.add_readable_rule_str()

print(f"\n📊 Rules generated: {len(rules_fp):,}")
print(f"📈 Average support: {rules_fp['support'].mean():.4f}")
print(f"📈 Average confidence: {rules_fp['confidence'].mean():.4f}")
print(f"📈 Average lift: {rules_fp['lift'].mean():.2f}")

# Xem thử
print("\n👀 First 10 rules:")
preview_cols = ['antecedents_str', 'consequents_str', 'support', 'confidence', 'lift']
display(rules_fp[preview_cols].head(10))


GENERATING ASSOCIATION RULES

Generating association rules (metric='lift', threshold=1.0)...
✓ Generated 0 association rules

📊 Rules generated: 0
📈 Average support: nan
📈 Average confidence: nan
📈 Average lift: nan

👀 First 10 rules:


,antecedents_str,consequents_str,support,confidence,lift


In [9]:
print("\n" + "="*60)
print("FILTERING RULES")
print("="*60)

# Lọc luật
rules_filtered_fp = miner.filter_rules(
    min_support=FILTER_MIN_SUPPORT,
    min_confidence=FILTER_MIN_CONF,
    min_lift=FILTER_MIN_LIFT,
    max_len_antecedents=FILTER_MAX_ANTECEDENTS,
    max_len_consequents=FILTER_MAX_CONSEQUENTS
)

print(f"\n✅ Filtering complete!")
print(f"📊 Rules before filtering: {len(rules_fp):,}")
print(f"📊 Rules after filtering: {len(rules_filtered_fp):,}")

# 🔒 FIX ZeroDivisionError
if len(rules_fp) > 0:
    percentage = len(rules_filtered_fp) / len(rules_fp) * 100
    print(f"📈 Percentage kept: {percentage:.1f}%")
else:
    print("📈 Percentage kept: 0.0% (No rules generated)")

# Top rules sau khi lọc
print("\n🏆 Top 10 filtered rules (by lift):")
if not rules_filtered_fp.empty:
    top_rules = rules_filtered_fp.head(10)
    for i, (_, rule) in enumerate(top_rules.iterrows(), 1):
        print(f"{i:2d}. {rule['antecedents_str']} → {rule['consequents_str']}")
        print(
            f"    Support: {rule['support']:.4f}, "
            f"Confidence: {rule['confidence']:.3f}, "
            f"Lift: {rule['lift']:.2f}"
        )
else:
    print("⚠️ Không có luật nào sau khi lọc.")



FILTERING RULES

Filtering rules with:
  - min_support >= 0.01
  - min_confidence >= 0.3
  - min_lift >= 1.2
  - antecedents length <= 2
  - consequents length <= 1
✓ After filtering: 0 rules

✅ Filtering complete!
📊 Rules before filtering: 0
📊 Rules after filtering: 0
📈 Percentage kept: 0.0% (No rules generated)

🏆 Top 10 filtered rules (by lift):
⚠️ Không có luật nào sau khi lọc.


In [10]:
import matplotlib.pyplot as plt
import networkx as nx

class DataVisualizer:
    def plot_top_rules_lift(self, rules_df, top_n=10, title=None):
        top_rules = rules_df.sort_values("lift", ascending=False).head(top_n)

        plt.figure(figsize=(10, 6))
        plt.barh(
            range(len(top_rules)),
            top_rules["lift"]
        )
        plt.yticks(
            range(len(top_rules)),
            top_rules["antecedents"].astype(str) + " → " +
            top_rules["consequents"].astype(str)
        )
        plt.xlabel("Lift")
        plt.title(title if title else "Top rules by Lift")
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.show()

    def plot_top_rules_confidence(self, rules_df, top_n=10, title=None):
        top_rules = rules_df.sort_values("confidence", ascending=False).head(top_n)

        plt.figure(figsize=(10, 6))
        plt.barh(
            range(len(top_rules)),
            top_rules["confidence"]
        )
        plt.yticks(
            range(len(top_rules)),
            top_rules["antecedents"].astype(str) + " → " +
            top_rules["consequents"].astype(str)
        )
        plt.xlabel("Confidence")
        plt.title(title if title else "Top rules by Confidence")
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.show()

    def plot_rules_support_confidence_scatter(self, rules_df, title=None):
        plt.figure(figsize=(8, 6))
        plt.scatter(
            rules_df["support"],
            rules_df["confidence"],
            alpha=0.6
        )
        plt.xlabel("Support")
        plt.ylabel("Confidence")
        plt.title(title if title else "Support vs Confidence")
        plt.tight_layout()
        plt.show()

    def plot_rules_network(self, rules_df, max_rules=30, title=None):
        G = nx.DiGraph()

        for _, row in rules_df.head(max_rules).iterrows():
            for a in row["antecedents"]:
                for c in row["consequents"]:
                    G.add_edge(a, c)

        plt.figure(figsize=(12, 8))
        pos = nx.spring_layout(G, seed=42)
        nx.draw(G, pos, with_labels=True, node_size=2000)
        plt.title(title if title else "Association Rules Network")
        plt.show()


In [11]:
print("\n" + "="*60)
print("VISUALIZATION")
print("="*60)

# Khởi tạo visualizer
visualizer = DataVisualizer()

# 9.1: Top rules by lift
if PLOT_TOP_LIFT and not rules_filtered_fp.empty:
    print("\n📊 Plotting top rules by LIFT...")
    visualizer.plot_top_rules_lift(
        rules_df=rules_filtered_fp,
        top_n=TOP_N_RULES,
        title=f"Top {TOP_N_RULES} Rules by Lift (FP-Growth)"
    )

# 9.2: Top rules by confidence
if PLOT_TOP_CONF and not rules_filtered_fp.empty:
    print("\n📊 Plotting top rules by CONFIDENCE...")
    visualizer.plot_top_rules_confidence(
        rules_df=rules_filtered_fp,
        top_n=TOP_N_RULES,
        title=f"Top {TOP_N_RULES} Rules by Confidence (FP-Growth)"
    )

# 9.3: Scatter plot
if PLOT_SCATTER and not rules_filtered_fp.empty:
    print("\n📊 Plotting scatter plot...")
    visualizer.plot_rules_support_confidence_scatter(
        rules_df=rules_filtered_fp,
        title="Support vs Confidence (FP-Growth)"
    )

# 9.4: Network graph
if PLOT_NETWORK and not rules_filtered_fp.empty:
    print("\n🕸️  Plotting network graph...")
    visualizer.plot_rules_network(
        rules_df=rules_filtered_fp,
        max_rules=min(TOP_N_RULES, 30),
        title="Association Rules Network (FP-Growth)"
    )

print("✅ Visualization complete!")



VISUALIZATION
✅ Visualization complete!


In [12]:
# Interactive plot với Plotly
if PLOT_PLOTLY_SCATTER and not rules_filtered_fp.empty:
    print("\n📈 Creating interactive Plotly visualization...")
    
    try:
        import plotly.express as px
        import plotly.graph_objects as go
        
        # Tạo interactive scatter
        fig = px.scatter(
            rules_filtered_fp.head(50),
            x='support',
            y='confidence',
            size='lift',
            color='lift',
            hover_name='rule_str',
            hover_data=['support', 'confidence', 'lift'],
            title='FP-Growth Rules: Support vs Confidence',
            labels={
                'support': 'Support',
                'confidence': 'Confidence',
                'lift': 'Lift'
            }
        )
        
        fig.update_layout(
            width=900,
            height=600,
            hovermode='closest'
        )
        
        fig.show()
        print("✅ Interactive plot created!")
        
    except ImportError:
        print("⚠️  Plotly not installed. Install with: pip install plotly")

In [13]:
print("\n" + "="*60)
print("SAVING RESULTS")
print("="*60)

# Điều chỉnh đường dẫn output
if not os.path.exists(RULES_OUTPUT_PATH):
    abs_output_path = os.path.join(project_root, "data", "processed", "rules_fpgrowth_filtered.csv")
    RULES_OUTPUT_PATH = abs_output_path

# Lưu rules
miner.save_rules(
    output_path=RULES_OUTPUT_PATH,
    rules_df=rules_filtered_fp
)

print(f"\n📁 Saved to: {RULES_OUTPUT_PATH}")

# Thông tin tóm tắt
print("\n📋 SUMMARY")
print("-" * 40)
print(f"Algorithm: FP-Growth")
print(f"Min Support: {MIN_SUPPORT}")
print(f"Frequent Itemsets: {len(frequent_itemsets_fp):,}")
print(f"Rules Generated: {len(rules_fp):,}")
print(f"Rules Filtered: {len(rules_filtered_fp):,}")
print(f"Top Lift: {rules_filtered_fp['lift'].max():.2f}")
print(f"Top Confidence: {rules_filtered_fp['confidence'].max():.3f}")
print(f"File Size: {os.path.getsize(RULES_OUTPUT_PATH) / 1024:.1f} KB")


SAVING RESULTS
✓ Rules saved to: data/processed/rules_fpgrowth_filtered.csv
  - Number of rules: 0

📁 Saved to: data/processed/rules_fpgrowth_filtered.csv

📋 SUMMARY
----------------------------------------
Algorithm: FP-Growth
Min Support: 0.05
Frequent Itemsets: 34
Rules Generated: 0
Rules Filtered: 0
Top Lift: nan
Top Confidence: nan
File Size: 0.2 KB


In [14]:
# ## 📊 BUSINESS INSIGHTS FROM FP-GROWTH

# ### 1. **Key Findings**
# - **Top Rules**: Danh sách các luật có lift cao nhất
# - **Strong Associations**: Các cặp sản phẩm thường mua cùng
# - **Product Bundles**: Gợi ý combo sản phẩm

# ### 2. **Recommendations**
# 1. **Cross-selling**: Trưng bày các sản phẩm có liên quan gần nhau
# 2. **Promotions**: Tạo combo giảm giá cho các itemsets phổ biến
# 3. **Inventory**: Quản lý tồn kho cho các sản phẩm trong cùng rules

# ### 3. **Next Steps**
# - So sánh với Apriori trong notebook riêng
# - Phân tích sensitivity với các min_support khác nhau
# - Áp dụng weighted association rules